# Formula 1 Performance Analytics

This project explores how Formula 1 race results are related to qualifying/start position, constructors, circuits, and seasons.

## 1. Abstract / Annotation

Formula 1 is a data-rich sport where race outcomes depend on many measurable factors: starting grid position, constructor performance, driver consistency, circuit type, and season. In this project, I analyze historical Formula 1 data to understand which factors are most strongly related to final race results.

The main focus is on position changes from the starting grid to the final race result, podium finishes, points, and constructor-level performance.

## 2. Imports and Data Loading

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

In [ ]:
DATA_DIR = Path("data")
csv_files = sorted(DATA_DIR.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError("No CSV files found. Download the dataset and put CSV files into the data/ folder.")

tables = {}
for path in csv_files:
    key = path.stem.replace("cleaned_", "")
    tables[key] = pd.read_csv(path)

list(tables.keys())

In [ ]:
for name, df in tables.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

## 3. Dataset Description

The dataset contains historical Formula 1 information. It is a composite dataset, meaning that the final analysis table is created by combining several CSV files: race results, races, drivers, constructors, and circuits.

Below I inspect the structure of the main tables, their columns, missing values, and data types.

In [ ]:
required_tables = ["results", "races", "drivers", "constructors", "circuits"]
missing_tables = [name for name in required_tables if name not in tables]

if missing_tables:
    print("Missing expected tables:", missing_tables)
else:
    print("All main tables are available.")

In [ ]:
for name in required_tables:
    if name in tables:
        print("\n" + "=" * 80)
        print(name.upper())
        display(tables[name].head())
        display(tables[name].info())

## 4. Build Main Analysis Table

I combine race results with season, driver, constructor, and circuit information.

In [ ]:
results = tables["results"].copy()
races = tables["races"].copy()
drivers = tables["drivers"].copy()
constructors = tables["constructors"].copy()
circuits = tables["circuits"].copy()

main = results.merge(races, on="raceId", how="left", suffixes=("", "_race"))
main = main.merge(drivers, on="driverId", how="left", suffixes=("", "_driver"))
main = main.merge(constructors, on="constructorId", how="left", suffixes=("", "_constructor"))
main = main.merge(circuits, on="circuitId", how="left", suffixes=("", "_circuit"))

main.shape

In [ ]:
main.head()

## 5. Data Cleanup

Even though the dataset is cleaned, I still check missing values, duplicates, and data types. For this project, rows without valid starting grid position or final race position are not useful for position-change analysis, so I remove them.

In [ ]:
main.isna().sum().sort_values(ascending=False).head(20)

In [ ]:
main.duplicated().sum()

In [ ]:
numeric_candidates = ["grid", "positionOrder", "points", "laps", "year", "round", "milliseconds", "fastestLapSpeed"]

for col in numeric_candidates:
    if col in main.columns:
        main[col] = pd.to_numeric(main[col], errors="coerce")

before_rows = len(main)
main_clean = main.dropna(subset=["grid", "positionOrder", "year", "points"]).copy()
main_clean = main_clean[main_clean["grid"] > 0].copy()
after_rows = len(main_clean)

print(f"Rows before cleanup: {before_rows}")
print(f"Rows after cleanup: {after_rows}")
print(f"Removed rows: {before_rows - after_rows}")

## 6. Data Transformation

I create new columns that are useful for analysis:

- `position_gain`: how many positions a driver gained or lost compared to the starting grid;
- `is_podium`: whether the driver finished in the top 3;
- `is_points_finish`: whether the driver scored more than 0 points.

In [ ]:
main_clean["position_gain"] = main_clean["grid"] - main_clean["positionOrder"]
main_clean["is_podium"] = main_clean["positionOrder"] <= 3
main_clean["is_points_finish"] = main_clean["points"] > 0

main_clean[["year", "grid", "positionOrder", "position_gain", "points", "is_podium"]].head()

## 7. Descriptive Statistics

The project requires descriptive statistics for at least 4 numerical fields. I use starting grid position, final position, points, laps, year, and position gain.

In [ ]:
stats_columns = ["grid", "positionOrder", "points", "laps", "year", "position_gain"]
stats_columns = [col for col in stats_columns if col in main_clean.columns]

main_clean[stats_columns].agg(["mean", "median", "std", "min", "max"]).T

## 8. Overview Plots

These plots show the general distribution and relationships of the main numerical fields.

In [ ]:
sns.histplot(main_clean["position_gain"], bins=40, kde=True)
plt.title("Distribution of Position Gain")
plt.xlabel("Position gain: grid position - final position")
plt.ylabel("Number of race entries")
plt.show()

In [ ]:
sns.scatterplot(data=main_clean, x="grid", y="positionOrder", alpha=0.35)
plt.title("Starting Grid Position vs Final Race Position")
plt.xlabel("Starting grid position")
plt.ylabel("Final race position")
plt.show()

In [ ]:
sns.lineplot(data=main_clean.groupby("year", as_index=False)["points"].mean(), x="year", y="points")
plt.title("Average Points per Race Entry by Season")
plt.xlabel("Season")
plt.ylabel("Average points")
plt.show()

In [ ]:
corr_columns = ["grid", "positionOrder", "points", "laps", "year", "position_gain"]
corr_columns = [col for col in corr_columns if col in main_clean.columns]

sns.heatmap(main_clean[corr_columns].corr(), annot=True, cmap="coolwarm", center=0)
plt.title("Correlation Between Main Numerical Fields")
plt.show()

## 9. More Detailed Overview

This section contains comparisons across constructors, seasons, and circuits.

In [ ]:
constructor_col = "name_constructor" if "name_constructor" in main_clean.columns else "name"

top_constructors = (
    main_clean.groupby(constructor_col)["points"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .index
)

constructor_summary = main_clean[main_clean[constructor_col].isin(top_constructors)].groupby(constructor_col).agg(
    avg_points=("points", "mean"),
    avg_position_gain=("position_gain", "mean"),
    podium_rate=("is_podium", "mean"),
    races=("raceId", "count")
).sort_values("avg_points", ascending=False)

constructor_summary

In [ ]:
constructor_summary["avg_position_gain"].sort_values().plot(kind="barh")
plt.title("Average Position Gain for Top Constructors")
plt.xlabel("Average position gain")
plt.ylabel("Constructor")
plt.show()

In [ ]:
main_clean["era"] = pd.cut(
    main_clean["year"],
    bins=[1949, 1979, 1999, 2013, 2023],
    labels=["1950-1979", "1980-1999", "2000-2013", "2014-2023"]
)

sns.boxplot(data=main_clean, x="era", y="position_gain")
plt.title("Position Gain by Formula 1 Era")
plt.xlabel("Era")
plt.ylabel("Position gain")
plt.show()

In [ ]:
era_corr = main_clean.groupby("era").apply(lambda x: x["grid"].corr(x["positionOrder"]))
era_corr = era_corr.rename("grid_position_correlation").reset_index()
era_corr

## 10. Hypothesis Checking

### Hypothesis 1

Starting grid position is strongly related to final race position, but the strength of this relationship differs across Formula 1 eras.

In [ ]:
sns.barplot(data=era_corr, x="era", y="grid_position_correlation")
plt.title("Correlation Between Grid Position and Final Position by Era")
plt.xlabel("Era")
plt.ylabel("Correlation")
plt.ylim(0, 1)
plt.show()

Interpretation: write here whether the hypothesis was supported by the data. Compare correlation values between eras.

### Hypothesis 2

Top constructors do not only score more points; they also have a higher podium rate and better average position gain.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

constructor_summary["avg_points"].sort_values().plot(kind="barh", ax=axes[0])
axes[0].set_title("Average Points by Constructor")
axes[0].set_xlabel("Average points")

constructor_summary["podium_rate"].sort_values().plot(kind="barh", ax=axes[1])
axes[1].set_title("Podium Rate by Constructor")
axes[1].set_xlabel("Podium rate")

plt.tight_layout()
plt.show()

Interpretation: write here whether the hypothesis was supported. Compare average points, podium rate, and position gain.

## 11. Discussion

Summarize the main results here:

- What numerical variables were analyzed?
- Which plots gave the clearest insight?
- Were the hypotheses supported?
- What are the limitations of the analysis?
- What could be improved in future work?